# omnidroid boot benchmark (real Android, headless)

Measures how fast **this machine's CPU and RAM** boot omnidroid's real-AOSP path -- real Android 15
(init, system_server, SurfaceFlinger...) running on omnidroid's own Linux ABI layer, arm64 code
translated to x86-64 as it runs -- and start the Roblox APK on it. No screen: the display is an
off-screen EGL pbuffer (Mesa llvmpipe on the CPU, or the NVIDIA GPU's EGL when there is one).

**No account is used.** No cookie is read, nothing signs in: the benchmark stops once the app's
first screen is up. (Colab exits from the US/EU; a cookie made in another country can be ended by
Roblox for good.)

What it times, from the log's own clock (`[t] +<secs>s` lines):

| run | what it is | timed |
|---|---|---|
| **new device** (once) | Android's first boot on an empty `/data`, `pm install` of the APK, the app's first start; then the device is saved | `boot_completed`, install done, app on screen |
| **saved device** (`SAVED_BOOTS` times) | a copy of the saved device boots, the app starts from its launcher -- what every later session does | copy, `boot_completed`, app on screen |

Each run also samples the machine every second: CPU busy (all cores), memory used, load.
Results are shown as a table and saved as JSON (Colab: Drive `MyDrive/omnidroid/bench/`), so runs
on different runtimes (CPU, T4, High-RAM, A100...) can be compared in the last cell.

**You need** the Roblox APK in Drive at `MyDrive/omnidroid/*.apk` (or uploaded to `/content`, or
`APK_PATH`). Everything else is downloaded: the source (`BRANCH` of `REPO_URL`), Rust, and Google's
public Android 15 emulator image (`arm64-v8a-35_r02.zip`, 769 MB), from which the system is built.

**Time.** First run: packages ~2 min, Rust ~1 min, the build ~20-40 min on Colab's 2 vCPUs, the
Android image ~10 min -- both cached in Drive (~1.5 GB), so later runs skip them. Then the new-device
run and each saved-device boot take minutes each. For scale, on a 16-core i7-13700F (Windows):
new device `boot_completed` ~150 s and the app on screen ~240 s, saved device ~145 s / ~190 s. The
same run pinned to 2 CPUs with llvmpipe (Colab's shape, on an old 4-core Linux box): new device
~220 s / ~390 s, saved device ~180 s / ~265 s. A run that fails prints what ended it and keeps its
logs in Drive (`bench/logs/`).

**Run it:** Runtime > Run all. On Colab, allow the Drive mount (the cache and the results live
there; without it everything is rebuilt each time).

In [ ]:
# @title Parameters
REPO_URL = "https://github.com/devflaryn/omnidroid"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
SOURCE_TARBALL = ""  # @param {type:"string"}
RUST_TOOLCHAIN = "1.89.0"  # @param {type:"string"}
USE_DRIVE_CACHE = True  # @param {type:"boolean"}
BUILD_JOBS = 0  # @param {type:"integer"}
APK_PATH = ""  # @param {type:"string"}
SAVED_BOOTS = 3  # @param {type:"integer"}
RUN_TIMEOUT_MINUTES = 45  # @param {type:"integer"}
APP_SETTLE_SECONDS = 20  # @param {type:"integer"}
RENDERER = "auto"  # @param ["auto", "gpu", "cpu"]
LABEL = ""  # @param {type:"string"}

# Every parameter can also come from the environment as OMNI_NB_<NAME>; the environment wins.
import os as _os
for _name in [n for n in list(globals()) if n.isupper() and not n.startswith("_")]:
    _raw = _os.environ.get("OMNI_NB_" + _name)
    if _raw is None:
        continue
    _old = globals()[_name]
    if isinstance(_old, bool):
        globals()[_name] = _raw.strip().lower() in ("1", "true", "yes", "on")
    elif isinstance(_old, int):
        globals()[_name] = int(_raw)
    else:
        globals()[_name] = _raw
    print(f"{_name} = {globals()[_name]!r}  (from OMNI_NB_{_name})")

## 1. This machine
What the benchmark runs on: CPU model, cores the notebook may use, memory, disk, `/dev/shm`, GPU.

In [ ]:
import collections, datetime, glob, hashlib, json, os, platform, re, shlex, shutil, signal
import subprocess, sys, threading, time
from pathlib import Path


def detect_platform():
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.path.isdir("/kaggle/input"):
        return "kaggle"
    return "linux"


PLATFORM = os.environ.get("OMNI_NB_PLATFORM") or detect_platform()
IS_ROOT = os.geteuid() == 0
HOME = Path.home()
SUDO = [] if IS_ROOT else (["sudo", "-n"] if shutil.which("sudo") and subprocess.run(
    ["sudo", "-n", "true"], capture_output=True).returncode == 0 else None)
# Fixed absolute paths: a cached build is only valid where it was built.
SRC = Path(os.environ.get("OMNI_NB_SRC") or ("/opt/omnidroid" if IS_ROOT else HOME / "omnidroid-nb" / "src"))
SYSROOT = Path(os.environ.get("OMNI_NB_SYSROOT") or ("/opt/aosp-35" if IS_ROOT else HOME / "omnidroid-nb" / "aosp-35"))
WORK = Path(os.environ.get("OMNI_NB_WORK") or {"colab": "/content/omnidroid-bench",
                                               "kaggle": "/kaggle/working/omnidroid-bench"}.get(PLATFORM, str(HOME / "omnidroid-nb" / "bench")))
WORK.mkdir(parents=True, exist_ok=True)
CARGO_HOME = Path(os.environ.get("CARGO_HOME", HOME / ".cargo"))
os_release = dict(l.rstrip("\n").split("=", 1) for l in open("/etc/os-release") if "=" in l)
DISTRO = os_release.get("ID", "?").strip('"') + os_release.get("VERSION_ID", "").strip('"')


def sh(cmd, env=None, cwd=None, check=True, show=None, quiet=False):
    """Run `cmd` (a list, or a bash string), streaming the lines `show` picks; the last 80 lines
    are printed if it fails."""
    shell = isinstance(cmd, str)
    proc = subprocess.Popen(cmd, shell=shell, executable="/bin/bash" if shell else None, env=env, cwd=cwd,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1)
    tail = collections.deque(maxlen=80)
    for line in proc.stdout:
        tail.append(line)
        if not quiet and (show is None or show(line)):
            print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        print("".join(tail))
        raise RuntimeError(f"exit code {code}: {cmd if shell else shlex.join(map(str, cmd))}")
    return code, "".join(tail)


def elapsed(t0):
    s = int(time.time() - t0)
    return f"{s // 60}:{s % 60:02d}"


def drive_dir():
    """MyDrive/omnidroid on Colab (mounted on first use), else a local directory."""
    if PLATFORM == "colab" and USE_DRIVE_CACHE:
        if not os.path.isdir("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive")
        if os.path.isdir("/content/drive/MyDrive"):
            d = Path("/content/drive/MyDrive/omnidroid")
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = Path("/kaggle/working") if PLATFORM == "kaggle" else WORK.parent / "cache"
    d.mkdir(parents=True, exist_ok=True)
    return d


def cpu_model():
    for line in open("/proc/cpuinfo"):
        if line.startswith("model name"):
            return line.split(":", 1)[1].strip()
    return platform.processor() or "?"


def cgroup_cpus():
    """The CPU quota of this container, if it has one (cgroup v2 cpu.max, v1 cfs)."""
    try:
        quota, period = open("/sys/fs/cgroup/cpu.max").read().split()
        return None if quota == "max" else int(quota) / int(period)
    except Exception:
        pass
    try:
        q = int(open("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read())
        p = int(open("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read())
        return None if q <= 0 else q / p
    except Exception:
        return None


def meminfo():
    m = {}
    for line in open("/proc/meminfo"):
        k, v = line.split(":", 1)
        m[k] = int(v.split()[0]) * 1024
    return m


gpu = []
if shutil.which("nvidia-smi"):
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    gpu = [l.strip() for l in out.stdout.splitlines() if l.strip()] if out.returncode == 0 else []
mem = meminfo()
shm = shutil.disk_usage("/dev/shm") if os.path.isdir("/dev/shm") else None
MACHINE = {
    "platform": PLATFORM, "os": os_release.get("PRETTY_NAME", "?"), "cpu": cpu_model(),
    "logical_cpus": os.cpu_count(), "usable_cpus": len(os.sched_getaffinity(0)), "cgroup_cpu_quota": cgroup_cpus(),
    "ram_gib": round(mem["MemTotal"] / 2**30, 1), "swap_gib": round(mem.get("SwapTotal", 0) / 2**30, 1),
    "disk_free_gib": round(shutil.disk_usage(WORK).free / 2**30, 1), "shm_gib": round(shm.total / 2**30, 1) if shm else 0,
    "gpu": "; ".join(gpu) or "none",
}
for k, v in MACHINE.items():
    print(f"{k:>17}: {v}")
if MACHINE["ram_gib"] < 10:
    print("⚠️  less than 10 GiB of RAM: one device holds ~4-5 GiB and its boot more; it may be killed")

## 2. Packages
Compilers and CMake (dynarmic, the arm64 -> x86-64 translator, is C++), the X11/ALSA headers the
platform layer links, Mesa's EGL/GLES (llvmpipe), `debugfs` and `fsck.erofs` to unpack the Android
image.

In [ ]:
t0 = time.time()
APT = """build-essential cmake ninja-build pkg-config git curl ca-certificates zstd xz-utils unzip
e2fsprogs erofs-utils python3 libasound2-dev libx11-dev libxi-dev libxfixes-dev
libegl1 libgles2 libegl-mesa0 libgl1-mesa-dri libglvnd0 libvulkan1""".split()
if DISTRO == "ubuntu22.04":
    APT.append("g++-12")  # 22.04's default g++ 11 is too old for dynarmic's C++20


def missing(pkgs):
    out = subprocess.run(["dpkg-query", "-W", "-f", "${Package} ${Status}\n", *pkgs], capture_output=True, text=True).stdout
    have = {l.split()[0] for l in out.splitlines() if l.endswith("install ok installed")}
    return [p for p in pkgs if p not in have]


todo = missing(APT)
if todo and SUDO is None:
    print("cannot install (no root, no passwordless sudo); missing:", " ".join(todo))
elif todo:
    print("installing:", " ".join(todo))
    env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
    sh(SUDO + ["apt-get", "update", "-qq"], env=env, quiet=True)
    sh(SUDO + ["apt-get", "install", "-y", "-qq", "--no-install-recommends", *todo], env=env,
       show=lambda l: l.startswith("E:"))
print("missing now:", missing(APT) or "nothing")
for mod in ("pandas", "matplotlib", "PIL"):
    try:
        __import__(mod)
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", {"PIL": "pillow"}.get(mod, mod)])

# Some Android APEX payloads are erofs: fsck.erofs must extract (erofs-utils >= 1.5). An older one
# (Ubuntu 22.04 ships 1.4) is replaced by one built from source.
FSCK_EROFS = shutil.which("fsck.erofs")
has_extract = FSCK_EROFS and "--extract" in subprocess.run([FSCK_EROFS, "--help"], capture_output=True, text=True).stdout + \
    subprocess.run([FSCK_EROFS, "--help"], capture_output=True, text=True).stderr
if not has_extract:
    print("building erofs-utils 1.8.2 (fsck.erofs --extract)")
    need = missing(["autoconf", "automake", "libtool", "liblz4-dev", "uuid-dev", "zlib1g-dev"])
    if need and SUDO is not None:
        sh(SUDO + ["apt-get", "install", "-y", "-qq", "--no-install-recommends", *need],
           env=dict(os.environ, DEBIAN_FRONTEND="noninteractive"), quiet=True)
    erofs_src = WORK / "erofs-utils"
    if not erofs_src.exists():
        sh(["git", "clone", "-q", "--depth", "1", "--branch", "v1.8.2", "https://github.com/erofs/erofs-utils", str(erofs_src)])
    sh("./autogen.sh && ./configure --prefix=$HOME/erofs/usr --disable-fuse && make -j$(nproc) && make install",
       cwd=erofs_src, quiet=True)
    FSCK_EROFS = str(HOME / "erofs/usr/bin/fsck.erofs")
print("fsck.erofs:", FSCK_EROFS, f"[{elapsed(t0)}]")

## 3. Rust
The pinned toolchain (`RUST_TOOLCHAIN`): the cached build is only valid for the compiler that made it.

In [ ]:
t0 = time.time()
os.environ["PATH"] = f"{CARGO_HOME / 'bin'}:{os.environ['PATH']}"
if not shutil.which("rustup"):
    sh(f"curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal "
       f"--default-toolchain {shlex.quote(RUST_TOOLCHAIN)} --no-modify-path", quiet=True)
sh(["rustup", "toolchain", "install", RUST_TOOLCHAIN, "--profile", "minimal"], quiet=True)
BUILD_ENV = dict(os.environ, RUSTUP_TOOLCHAIN=RUST_TOOLCHAIN, CARGO_TERM_COLOR="never")
BUILD_ENV.pop("OMNIDROID_DYNARMIC_BUILD_DIR", None)
if BUILD_JOBS > 0:
    BUILD_ENV["CARGO_BUILD_JOBS"] = str(BUILD_JOBS)
if DISTRO == "ubuntu22.04" and shutil.which("g++-12"):
    BUILD_ENV.update(CC="gcc-12", CXX="g++-12")
print(subprocess.run(["cargo", "--version"], env=BUILD_ENV, capture_output=True, text=True).stdout.strip(), f"[{elapsed(t0)}]")

## 4. Source and build
Clones `BRANCH`, then builds the runtime and the session (`omni-linux`'s `r_roblox`, with its
`omni-linux-run`). With the Drive cache, a build made earlier for the same commit, compiler and OS
is restored instead (seconds); a fresh one is saved there.

In [ ]:
t0 = time.time()
CACHE = drive_dir() / "cache"
CACHE.mkdir(parents=True, exist_ok=True)


def git_head(path):
    out = subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"], capture_output=True, text=True)
    return out.stdout.strip() if out.returncode == 0 else None


if SOURCE_TARBALL:
    COMMIT = "tarball-" + hashlib.sha256(open(SOURCE_TARBALL, "rb").read(1 << 24)).hexdigest()[:12]
else:
    out = subprocess.run(["git", "ls-remote", REPO_URL, f"refs/heads/{BRANCH}"], capture_output=True, text=True, timeout=120)
    if out.returncode != 0 or not out.stdout.strip():
        raise SystemExit(f"branch {BRANCH!r} not found at {REPO_URL}: {out.stderr.strip()[:300]}")
    COMMIT = out.stdout.split()[0]
BUILD_CACHE = CACHE / f"omnidroid-aosp-build-{COMMIT[:12]}-rust{RUST_TOOLCHAIN}-{DISTRO}.tar.zst"
RESTORED = False
if git_head(SRC) == COMMIT or (SOURCE_TARBALL and (SRC / "Cargo.toml").exists()):
    print(f"{SRC} is already at {COMMIT[:12]}")
elif USE_DRIVE_CACHE and BUILD_CACHE.exists():
    shutil.rmtree(SRC, ignore_errors=True)
    print(f"restoring {BUILD_CACHE.name} ({BUILD_CACHE.stat().st_size / 2**30:.2f} GiB)")
    sh(["tar", "-I", "zstd -d -T0", "-xf", str(BUILD_CACHE), "-C", "/"])
    RESTORED = True
else:
    shutil.rmtree(SRC, ignore_errors=True)
    SRC.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_TARBALL:
        SRC.mkdir()
        sh(["tar", "-xf", SOURCE_TARBALL, "-C", str(SRC), "--strip-components=1"])
    else:
        print(f"cloning {REPO_URL} {BRANCH} ({COMMIT[:12]})")
        sh(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(SRC)])
print(f"source {SRC} at {COMMIT[:12]} [{elapsed(t0)}]")

t1 = time.time()
print("building (the first time: ~20-40 min on 2 vCPUs)")
sh(["cargo", "test", "--release", "-p", "omni-linux", "--test", "r_roblox", "--no-run"], env=BUILD_ENV, cwd=SRC,
   show=lambda l: l.lstrip().startswith(("Compiling omni", "Compiling dynarmic", "Finished", "error")))
out = subprocess.run(["cargo", "test", "--release", "-p", "omni-linux", "--test", "r_roblox", "--no-run",
                      "--message-format=json"], env=BUILD_ENV, cwd=SRC, capture_output=True, text=True)
R_ROBLOX = None
for line in out.stdout.splitlines():
    try:
        msg = json.loads(line)
    except ValueError:
        continue
    if msg.get("reason") == "compiler-artifact" and msg.get("target", {}).get("name") == "r_roblox" and msg.get("executable"):
        R_ROBLOX = msg["executable"]
if not R_ROBLOX:
    print(out.stderr[-4000:])
    raise SystemExit("the build failed (above)")
print(f"built [{elapsed(t1)}]: {R_ROBLOX}")
FRESH_BUILD = not RESTORED and time.time() - t1 > 60
if USE_DRIVE_CACHE and FRESH_BUILD and not SOURCE_TARBALL:
    t2 = time.time()
    rel = str(SRC).lstrip("/")
    sh(["tar", "-I", "zstd -T0 -3", "-cf", str(BUILD_CACHE) + ".part", "-C", "/", f"--exclude={rel}/target/debug",
        f"--exclude={rel}/target/release/incremental", rel])
    os.replace(str(BUILD_CACHE) + ".part", BUILD_CACHE)
    print(f"build cached: {BUILD_CACHE} ({BUILD_CACHE.stat().st_size / 2**30:.2f} GiB) [{elapsed(t2)}]")

## 5. The Android system
Android 15's system is Google's public emulator image, `arm64-v8a-35_r02.zip` (the SDK's "Default
Android System Image", sha1 `2026a064...`), unpacked by `tools/make_sysroot.py` into omnidroid's
sysroot (every file by content, a manifest, and owners/labels). ~10 min the first time, then cached
in Drive (~1 GB).

In [ ]:
t0 = time.time()
IMAGE_URL = "https://dl.google.com/android/repository/sys-img/android/arm64-v8a-35_r02.zip"
IMAGE_SHA1 = "2026a06409db630b56711afdbffb457c1dbaed49"
SYSROOT_CACHE = CACHE / "aosp-35-sysroot-r02.tar.zst"
ready = (SYSROOT / "sysroot.manifest").exists() and (SYSROOT / "sysroot.meta").exists()
if ready:
    print(f"sysroot at {SYSROOT}")
elif USE_DRIVE_CACHE and SYSROOT_CACHE.exists():
    print(f"restoring {SYSROOT_CACHE.name} ({SYSROOT_CACHE.stat().st_size / 2**30:.2f} GiB)")
    SYSROOT.mkdir(parents=True, exist_ok=True)
    sh(["tar", "-I", "zstd -d -T0", "-xf", str(SYSROOT_CACHE), "-C", str(SYSROOT)])
else:
    zip_path = WORK / "arm64-v8a-35_r02.zip"
    if not zip_path.exists() or hashlib.sha1(zip_path.read_bytes()).hexdigest() != IMAGE_SHA1:
        print("downloading the Android 15 image (769 MB)")
        sh(["curl", "-fL", "--retry", "3", "-o", str(zip_path), IMAGE_URL], quiet=True)
    digest = hashlib.sha1(zip_path.read_bytes()).hexdigest()
    if digest != IMAGE_SHA1:
        raise SystemExit(f"{zip_path}: sha1 {digest}, expected {IMAGE_SHA1}")
    shutil.rmtree(SYSROOT, ignore_errors=True)
    env = dict(os.environ, PATH=f"{Path(FSCK_EROFS).parent}:/usr/sbin:/sbin:{os.environ['PATH']}", TMPDIR=str(WORK / "tmp"))
    (WORK / "tmp").mkdir(exist_ok=True)
    tool = str(SRC / "tools" / "make_sysroot.py")
    print("building the sysroot")
    sh([sys.executable, tool, "--zip", str(zip_path), "--out", str(SYSROOT)], env=env, show=lambda l: "sha256" in l or "error" in l.lower())
    sh([sys.executable, tool, "--zip", str(zip_path), "--out", str(SYSROOT), "--meta"], env=env, show=lambda l: "sha256" in l or "entries" in l)
    zip_path.unlink()
    if USE_DRIVE_CACHE:
        sh(["tar", "-I", "zstd -T0 -3", "-cf", str(SYSROOT_CACHE) + ".part", "-C", str(SYSROOT), "."])
        os.replace(str(SYSROOT_CACHE) + ".part", SYSROOT_CACHE)
        print(f"sysroot cached: {SYSROOT_CACHE} ({SYSROOT_CACHE.stat().st_size / 2**30:.2f} GiB)")
print(open(SYSROOT / "sysroot.manifest").readline().strip(), f"[{elapsed(t0)}]")

## 6. The APK
Found in Drive (`MyDrive/omnidroid/*.apk`), `/content`, or `APK_PATH`, and copied beside the run.
No account is used, so any Roblox build will do; the stock one shows "Upgrade required" once it
reaches Roblox's servers, which is after everything timed here.

In [ ]:
def find_apk():
    if APK_PATH:
        return Path(APK_PATH)
    pats = [str(drive_dir() / "*.apk"), "/content/drive/MyDrive/*.apk", "/content/*.apk", "/kaggle/input/*/*.apk",
            str(WORK / "*.apk"), str(HOME / "*.apk")]
    found = [p for pat in pats for p in sorted(glob.glob(pat)) if "/apk/" not in p]
    if not found:
        raise SystemExit("No APK. Put the Roblox APK in Drive MyDrive/omnidroid/ (or /content), or set APK_PATH.")
    return Path(found[0])


src_apk = find_apk()
(WORK / "apk").mkdir(exist_ok=True)
APK = WORK / "apk" / src_apk.name
if not APK.exists() or APK.stat().st_size != src_apk.stat().st_size:
    shutil.copyfile(src_apk, APK)
APK_SHA256 = hashlib.sha256(APK.read_bytes()).hexdigest()
print(f"APK {src_apk} ({APK.stat().st_size / 2**20:.0f} MiB) sha256 {APK_SHA256[:16]}...")

## 7. Graphics
The device's GPU backend is GL (`OMNI_GPU=gl`): the guest's GLES goes to the host's GLES through EGL
pbuffers, no window system. With an NVIDIA GPU whose EGL library the runtime exposes, that is the
GPU; otherwise Mesa llvmpipe on the CPU (which then shares the CPU with the boot being timed).

In [ ]:
def find_lib(name):
    for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout.splitlines():
        if line.strip().startswith(name + " ") and "x86-64" in line:
            return line.split("=>")[-1].strip()
    for d in ("/usr/lib64-nvidia", "/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu"):
        if os.path.exists(f"{d}/{name}"):
            return f"{d}/{name}"
    return None


nvidia_egl = find_lib("libEGL_nvidia.so.0") if MACHINE["gpu"] != "none" else None
USE_GPU = RENDERER != "cpu" and bool(nvidia_egl)
GFX_ENV = {"OMNI_GPU": "gl"}
mesa_json = next(iter(glob.glob("/usr/share/glvnd/egl_vendor.d/*mesa*.json")), None)
if USE_GPU:
    vendor = WORK / "egl" / "10_nvidia.json"
    vendor.parent.mkdir(exist_ok=True)
    vendor.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": nvidia_egl}}))
    GFX_ENV["__EGL_VENDOR_LIBRARY_FILENAMES"] = ":".join(filter(None, [str(vendor), mesa_json]))
    GFX_ENV["LD_LIBRARY_PATH"] = ":".join(filter(None, [os.path.dirname(nvidia_egl), os.environ.get("LD_LIBRARY_PATH")]))
else:
    GFX_ENV.update(LIBGL_ALWAYS_SOFTWARE="1", GALLIUM_DRIVER="llvmpipe")
    if mesa_json:
        GFX_ENV["__EGL_VENDOR_LIBRARY_FILENAMES"] = mesa_json
MACHINE["renderer"] = "NVIDIA EGL" if USE_GPU else "Mesa llvmpipe (CPU)"
if RENDERER == "gpu" and not USE_GPU:
    print("RENDERER='gpu', but no NVIDIA EGL here: llvmpipe")
print("renderer:", MACHINE["renderer"])

## 8. Run the benchmark
Each run is omnidroid's real-AOSP session (`r_roblox`) with no account and no place, its device
kept in `golden/` (`OMNI_R_GOLDEN`). The first run makes the device (and times that), saves it and
boots it again; every later run boots a copy. A run is stopped (its `stop` file: the device shuts
down cleanly) once the app has been on screen `APP_SETTLE_SECONDS`.

In [ ]:
GOLDEN = WORK / "golden"
RUNS_DIR = WORK / "runs"
RUNS_DIR.mkdir(exist_ok=True)
T_RE = re.compile(r"^\[t\] \+([0-9.]+)s (.*)")
MARKS = [("boot_completed", "[r] boot_completed=1"), ("installed", "[r] pm install:"), ("app_on_screen", "[r] am start:"),
         ("device_saved", "[r] device quiet")]


def milestones(log):
    """{name: seconds from that boot's start} from a boot's log."""
    found = {}
    if not log.exists():
        return found
    with open(log, errors="replace") as f:
        for line in f:
            m = T_RE.match(line)
            if not m:
                continue
            for name, marker in MARKS:
                if name not in found and marker in m.group(2):
                    found[name] = float(m.group(1))
            if "GUEST THREAD DIED" in line:
                found.setdefault("guest_thread_died", float(m.group(1)))
    return found


class Sampler(threading.Thread):
    """CPU busy (all cores), memory used and load, every second, into `rows`."""
    def __init__(self):
        super().__init__(daemon=True)
        self.rows, self.stop = [], threading.Event()

    def run(self):
        def cpu():
            f = [int(x) for x in open("/proc/stat").readline().split()[1:]]
            return sum(f), f[3] + f[4]
        t0, (tot0, idle0) = time.time(), cpu()
        while not self.stop.wait(1):
            tot, idle = cpu()
            busy = 100 * (1 - (idle - idle0) / max(1, tot - tot0))
            tot0, idle0 = tot, idle
            m = meminfo()
            self.rows.append((time.time() - t0, busy, (m["MemTotal"] - m["MemAvailable"]) / 2**30, os.getloadavg()[0]))


def stats(rows, t_from, t_to):
    part = [r for r in rows if t_from <= r[0] <= t_to] or rows
    if not part:
        return {}
    return {"cpu_busy_pct": round(sum(r[1] for r in part) / len(part), 1), "mem_peak_gib": round(max(r[2] for r in part), 2),
            "load_avg": round(sum(r[3] for r in part) / len(part), 2)}


def diagnose(log):
    """What ended a boot, from its log: each fatal exception with its stack, the lines before a
    host crash (dynarmic's "Unhandled SIGSEGV"), watchdog kills, dead guest threads, killed processes."""
    lines = open(log, errors="replace").read().splitlines()
    last_t = next((l for l in reversed(lines) if l.startswith("[t] +")), "")
    print(f"  -- {log.name}: {len(lines)} lines, last time mark {last_t.split(' ')[1] if last_t else '?'}")
    shown = 0
    for i, line in enumerate(lines):
        if shown >= 6:
            break
        if "FATAL EXCEPTION" in line:
            block = [l for l in lines[i:i + 40] if l.startswith("E/AndroidRuntime")][:25]
        elif "Unhandled SIGSEGV" in line or "Unhandled SIGBUS" in line:
            block = lines[max(0, i - 25):i + 3]
        elif "WATCHDOG KILLING" in line or "GUEST THREAD DIED" in line:
            block = lines[max(0, i - 5):i + 20]
        elif line.startswith("omni-linux: Killed") or line.startswith("omni-linux-run:"):
            block = lines[i:i + 6]
        else:
            continue
        shown += 1
        print("  " + "\n  ".join(l[:220] for l in block if "present fence" not in l))
        print("  --")


def keep_logs(label, paths):
    """A failed run's logs, compressed, beside the results (Colab: Drive), for whoever looks next."""
    dest = drive_dir() / "bench" / "logs" / f"{started:%Y%m%d-%H%M}-{label}"
    dest.mkdir(parents=True, exist_ok=True)
    for p in paths:
        if p.exists():
            subprocess.run(["zstd", "-q", "-f", "-10", str(p), "-o", str(dest / (p.name + ".zst"))])
    print(f"  logs kept in {dest}")


def run_session(label):
    """One r_roblox session; returns its figures."""
    inst = RUNS_DIR / label
    for p in (inst, Path(f"{inst}-shots")):
        shutil.rmtree(p, ignore_errors=True)
    for p in RUNS_DIR.glob(f"{label}.*"):
        p.unlink()
    for f in glob.glob("/dev/shm/omni-shm-*") + glob.glob(str(WORK / "tmp" / "omni-shm-*")):
        try:
            os.unlink(f)  # a finished session's graphics buffers
        except OSError:
            pass
    env = dict(BUILD_ENV, **GFX_ENV)
    for k in ("OMNI_R_COOKIE", "OMNI_R_PLACE", "OMNI_WINDOW", "DISPLAY", "WAYLAND_DISPLAY"):
        env.pop(k, None)
    env.update(OMNI_SYSROOT=str(SYSROOT), OMNI_TEST_APK=str(APK), OMNI_R_KIOSK="1", OMNI_R_MINUTES=str(2 * RUN_TIMEOUT_MINUTES),
               OMNI_R_GOLDEN=str(GOLDEN), OMNI_R_INSTANCE=str(inst), OMNI_SCREENSHOT=f"{inst}.png", TMPDIR=str(WORK / "tmp"))
    (WORK / "tmp").mkdir(exist_ok=True)
    if MACHINE["shm_gib"] < 2:  # graphics buffers: /dev/shm is RAM; a small one (containers) cannot hold them
        env["OMNI_SHM_DIR"] = str(WORK / "tmp")
    fresh = not (GOLDEN.exists() and any(GOLDEN.glob("*/ready")))
    out_path = RUNS_DIR / f"{label}.out"
    sampler = Sampler()
    sampler.start()
    t0 = time.time()
    proc = subprocess.Popen([R_ROBLOX, "--ignored", "--nocapture", "--exact", "the_apk_is_installed_started_and_draws"],
                            cwd=SRC / "crates" / "omni-linux", env=env, stdout=open(out_path, "w"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    log, first_log = Path(f"{inst}.log"), Path(f"{inst}.1.log")
    print(f"{label}: {'a new device, saved, then booted again' if fresh else 'a copy of the saved device'}  (log {log})")
    shown, on_screen_at, result, second_at = set(), None, "ok", None
    limit = RUN_TIMEOUT_MINUTES * 60 * (2 if fresh else 1)  # a new device boots twice
    while True:
        time.sleep(2)
        if first_log.exists() and second_at is None:
            second_at = time.time() - t0  # the saved device's boot starts (within the 2 s poll)
        boots = ([("new device", milestones(first_log)), ("saved device", milestones(log))] if first_log.exists()
                 else [("new device" if fresh else "saved device", milestones(log))])
        for which, ms in boots:
            for name, secs in ms.items():
                if (which, name) not in shown:
                    shown.add((which, name))
                    print(f"  [{elapsed(t0)}] {which}: {name} at +{secs:.1f}s")
        last = boots[-1]
        done = last[0] == "saved device" and "app_on_screen" in last[1]
        if done and on_screen_at is None:
            on_screen_at = time.time()
        if on_screen_at and time.time() - on_screen_at >= APP_SETTLE_SECONDS:
            break
        if proc.poll() is not None:
            result = f"the session ended early (exit {proc.returncode})"
            break
        if time.time() - t0 > limit:
            result = f"timeout after {limit // 60} min"
            break
    shot = RUNS_DIR / f"{label}-screen.png"
    if Path(f"{inst}.png").exists():
        shutil.copyfile(f"{inst}.png", shot)
    if proc.poll() is None:
        stop = inst / "data" / "local" / "tmp" / "stop"
        stop.parent.mkdir(parents=True, exist_ok=True)
        stop.write_text("1")
        try:
            proc.wait(90)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
    sampler.stop.set()
    wall = time.time() - t0
    text = out_path.read_text(errors="replace")
    copy = re.search(r"\[r\] device from .*?: (\d+) MiB in (\d+) ms", text)
    saved = re.search(r"\[r\] device saved as .*?: (\d+) MiB in (\d+) ms", text)
    if result != "ok":
        print("  !!", result)
        for l in (first_log, log):
            if l.exists():
                diagnose(l)
        keep_logs(label, [first_log, log, out_path])
    rows = []
    for which, ms in boots:
        row = {"run": label, "boot": which, **{k: v for k, v in ms.items() if k != "device_saved"}}
        # The sampler's clock starts with the run; the second boot of a new-device run starts later.
        start = second_at if (which == "saved device" and second_at is not None) else 0.0
        row.update(stats(sampler.rows, start, start + ms.get("app_on_screen", ms.get("boot_completed", 1e9))))
        if which == "saved device" and copy:
            row["copy_s"] = int(copy.group(2)) / 1000
        if which == "new device" and saved:
            row["save_s"] = int(saved.group(2)) / 1000
        row["result"] = result
        rows.append(row)
    print(f"  {label} done in {wall / 60:.1f} min: {result}")
    return rows, sampler.rows, shot


ALL_ROWS, TRACES, SHOTS = [], {}, {}
started = datetime.datetime.now(datetime.timezone.utc)
labels = ["run1"] + [f"run{i + 2}" for i in range(SAVED_BOOTS - 1)]
if GOLDEN.exists() and not any(GOLDEN.glob("*/ready")):
    shutil.rmtree(GOLDEN)
for label in labels:
    rows, trace, shot = run_session(label)
    ALL_ROWS += rows
    TRACES[label], SHOTS[label] = trace, shot
    if any(r["result"] != "ok" for r in rows) and not any(GOLDEN.glob("*/ready")):
        print("no saved device to boot again: stopping here")
        break

## 9. Results
Seconds from each boot's own start (`boot_completed`: Android's `sys.boot_completed=1`;
`app_on_screen`: `am start -W` of the app's launcher returned, its first screen drawn). CPU busy is
the average over all cores from the boot's start to the app on screen; 100% means every core busy.
Saved as JSON beside the cache so other runtimes can be compared below.

In [ ]:
import pandas as pd
from IPython.display import Image as IPyImage, display

df = pd.DataFrame(ALL_ROWS)
cols = [c for c in ["run", "boot", "copy_s", "boot_completed", "installed", "app_on_screen", "save_s", "cpu_busy_pct",
                    "mem_peak_gib", "load_avg", "result"] if c in df.columns]
display(df[cols])
saved = df[(df["boot"] == "saved device") & df.get("app_on_screen", pd.Series(dtype=float)).notna()] if "app_on_screen" in df else df.iloc[0:0]
summary = {}
if len(saved):
    summary = {"saved_boot_completed_median_s": float(saved["boot_completed"].median()),
               "saved_app_on_screen_median_s": float(saved["app_on_screen"].median()),
               "saved_boots": int(len(saved))}
new = df[df["boot"] == "new device"]
if len(new) and "boot_completed" in new:
    summary["new_boot_completed_s"] = float(new["boot_completed"].iloc[0])
    if "app_on_screen" in new:
        summary["new_app_on_screen_s"] = float(new["app_on_screen"].iloc[0])
print(json.dumps(summary, indent=1))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for label, trace in TRACES.items():
    if trace:
        ax[0].plot([r[0] for r in trace], [r[1] for r in trace], label=label, lw=1)
        ax[1].plot([r[0] for r in trace], [r[2] for r in trace], label=label, lw=1)
ax[0].set(title="CPU busy, all cores (%)", xlabel="s from the run's start", ylim=(0, 105))
ax[1].set(title="memory used (GiB)", xlabel="s from the run's start")
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

last = next((s for s in reversed(list(SHOTS.values())) if s.exists()), None)
if last:
    print("the device's screen when the last run was stopped:")
    display(IPyImage(filename=str(last), width=640))

BENCH = drive_dir() / "bench"
BENCH.mkdir(exist_ok=True)
tag = LABEL or f"{MACHINE['cpu'].split('@')[0].strip().replace(' ', '_')}-{MACHINE['usable_cpus']}cpu-{MACHINE['ram_gib']:.0f}g"
out = BENCH / f"{started:%Y%m%d-%H%M}-{re.sub(r'[^A-Za-z0-9_.-]+', '_', tag)}.json"
out.write_text(json.dumps({"when": started.isoformat(), "commit": COMMIT, "apk_sha256": APK_SHA256, "machine": MACHINE,
                           "summary": summary, "rows": ALL_ROWS}, indent=1, default=str))
print("saved:", out)

## 10. Compare runtimes
Every result saved so far (this runtime and others), one row each.

In [ ]:
rows = []
for f in sorted((drive_dir() / "bench").glob("*.json")):
    d = json.loads(f.read_text())
    m, s = d["machine"], d.get("summary", {})
    rows.append({"when": d["when"][:16], "cpu": m["cpu"][:40], "cpus": m["usable_cpus"], "ram_gib": m["ram_gib"],
                 "renderer": m.get("renderer"), "new_boot_s": s.get("new_boot_completed_s"),
                 "saved_boot_s": s.get("saved_boot_completed_median_s"), "saved_app_s": s.get("saved_app_on_screen_median_s"),
                 "commit": d.get("commit", "")[:8]})
display(pd.DataFrame(rows))